In [85]:
import pandas as pd
import json

In [86]:
with open("2024-1.json", "r", encoding="utf-8") as file:
    data = json.load(file)

In [87]:
# Prepare data for tables
courses_data = []
sections_data = []
teachers_data = []
schedule_data = []
section_teachers_data = []
section_schedule_data = []
quotas_data = []

# Iterate through courses and sections to extract relevant data
for course_key, course_value in data.items():
    # Course table data
    courses_data.append({
        "sigle": course_value["sigle"],
        "name": course_value["name"],
        "credits": course_value["credits"],
        "school": course_value["school"],
        "area": course_value["area"],
        "category": course_value["category"]
    })

    # Process sections
    for section_key, section_value in course_value["sections"].items():
        section_sigle = course_value["sigle"]
        section_number = section_value["section"]
        section_year = section_value["year"]
        section_semester = section_value["semester"]
        section_format = section_value["format"]
        section_is_english = section_value["is_english"]
        section_is_removable = section_value["is_removable"]
        section_is_special = section_value["is_special"]

        # Section table data
        sections_data.append({
            "sigle": section_sigle,
            "number": section_number,
            "year": section_year,
            "semester": section_semester,
            "format": section_format,
            "is_english": section_is_english,
            "is_removable": section_is_removable,
            "is_special": section_is_special
        })

        # Teachers data
        for teacher in section_value["teachers"]:
            teachers_data.append({
                "name": teacher
            })
            # SectionTeacher relationship data
            section_teachers_data.append({
                "section_sigle": section_sigle,
                "section_number": section_number,
                "year": section_year,
                "semester": section_semester,
                "teacher_name": teacher
            })

        # Schedule data
        for day_block, schedule_info in section_value["schedule"].items():
            schedule_data.append({
                "day_block": day_block,
                "type": schedule_info["type"],
                "place": schedule_info["place"],
                "campus": schedule_info["campus"]

            })
            # SectionSchedule relationship data
            section_schedule_data.append({
                "section_sigle": section_sigle,
                "section_number": section_number,
                "year": section_year,
                "semester": section_semester,
                "day_block": day_block,
                "place": schedule_info["place"]
            })

        # Quota data (empty in this case, can be expanded as needed)
        for quota_type, count in section_value["quota"].items():
            quotas_data.append({
                "section_sigle": section_sigle,
                "section_number": section_number,
                "section_year": section_year,
                "section_semester": section_semester,
                "type": quota_type,
                "count": count
            })

# Convert lists to pandas DataFrames
courses_df = pd.DataFrame(courses_data)
sections_df = pd.DataFrame(sections_data)
teachers_df = pd.DataFrame(teachers_data)
schedule_df = pd.DataFrame(schedule_data)
section_teachers_df = pd.DataFrame(section_teachers_data)
section_schedule_df = pd.DataFrame(section_schedule_data)
quotas_df = pd.DataFrame(quotas_data)


In [88]:
from dotenv import load_dotenv
import os


load_dotenv()


POSTGRES_USER = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_DB = os.getenv("POSTGRES_DB")
POSTGRES_HOST = os.getenv("POSTGRES_HOST")
POSTGRES_PORT = os.getenv("POSTGRES_PORT")

assert POSTGRES_USER is not None, "Environment variable POSTGRES_USER is not set."
assert POSTGRES_PASSWORD is not None, "Environment variable POSTGRES_PASSWORD is not set."
assert POSTGRES_DB is not None, "Environment variable POSTGRES_DB is not set."
assert POSTGRES_HOST is not None, "Environment variable POSTGRES_HOST is not set."
assert POSTGRES_PORT is not None, "Environment variable POSTGRES_PORT is not set."


In [89]:
import psycopg2
from psycopg2 import sql
import pandas as pd

# Configuración de la conexión a PostgreSQL
conn = psycopg2.connect(
    dbname=POSTGRES_DB,  # Cambia por el nombre de tu base de datos
    user=POSTGRES_USER,                 # Cambia por tu usuario de PostgreSQL
    password=POSTGRES_PASSWORD,          # Cambia por tu contraseña
    host=POSTGRES_HOST,               # Cambia por el host si es necesario
    port=POSTGRES_PORT                     # Cambia el puerto si es necesario
)

# Crear un cursor para ejecutar las consultas



In [91]:
cursor = conn.cursor()


def insert_data_from_df(df, table_name, conflict_columns):
    try:
        for _, row in df.iterrows():
            # Genera la consulta de inserción con parámetros
            columns = ', '.join(df.columns)
            values = tuple(row)
            placeholders = ', '.join(['%s'] * len(row))  # Usar parámetros en lugar de concatenar
            query = f"""
                INSERT INTO {table_name} ({columns}) 
                VALUES ({placeholders})
                
                ;
            """
            cursor.execute(query, values)
            conn.commit()  # Realiza commit al final de las inserciones
    except Exception as e:
        conn.rollback()  # Deshace los cambios en caso de error
        print(f"Error al insertar datos en {table_name}: {e}")

# Insertar los datos de las tablas con los conflict_columns adecuados
insert_data_from_df(courses_df, "Course", ['course_id'])
insert_data_from_df(sections_df, "Section", ['section_id'])
insert_data_from_df(teachers_df, "Teacher", ['teacher_id'])
insert_data_from_df(schedule_df, "Schedule", ['schedule_id'])
insert_data_from_df(section_teachers_df, "SectionTeacher", ['section_id', 'teacher_id'])
insert_data_from_df(section_schedule_df, "SectionSchedule", ['section_id', 'schedule_id'])
insert_data_from_df(quotas_df, "Quota", ['quota_id'])

# Cerrar el cursor y la conexión
cursor.close()

print("Datos insertados correctamente en PostgreSQL.")

Error al insertar datos en Teacher: duplicate key value violates unique constraint "teacher_pkey"
DETAIL:  Key (name)=(Claudia Echenique) already exists.

Error al insertar datos en Schedule: duplicate key value violates unique constraint "schedule_pkey"
DETAIL:  Key (day_block, place, campus)=(m8, SIN SALA, Oriente) already exists.

Error al insertar datos en SectionTeacher: insert or update on table "sectionteacher" violates foreign key constraint "sectionteacher_teacher_name_fkey"
DETAIL:  Key (teacher_name)=(Andres Grumann) is not present in table "teacher".

Datos insertados correctamente en PostgreSQL.


In [ ]:
conn.close()
